<h3>Imports & setup</h3>

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import regexp_replace 
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists silver")
silver_run_id = str(uuid.uuid4())
print("current Silver Run Id :", silver_run_id)

# <h3> Step 2 : Sliver Control table </h3>

In [0]:
spark.sql(''' 
          create table if not exists silver.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingestion_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          ''')

### step 3 : Helper Functions

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("t")
         .merge(df_source.alias("s"),f"t.{join_key} = s.{join_key}"
        ).whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingestion_at(entity_name:str):
    ctrl = (spark.table("silver.processing_control")\
        .filter(
            (F.col("layer") == "silver")
             & (F.col("entity_name") == entity_name)
             & (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc()).limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None,None
    return rows[0]['last_processed_bronze_ingestion_at'],rows[0]['last_processed_bronze_run_id']

In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingestion_at,rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "silver",
          entity_name,
          last_processed_bronze_run_id,
          last_processed_bronze_ingestion_at,
          int(rows_merged),
          "success",
          silver_run_id,
          datetime.now()
          )],
        schema = """ 
                layer string,
                entity_name string,
                last_processed_bronze_run_id string,
                last_processed_bronze_ingestion_at timestamp,
                rows_merged bigint,
                run_status string,
                silver_run_id string,
                updated_at timestamp
                """)
    
    dt = DeltaTable.forName(spark, "silver.processing_control")
    (dt.alias("t")
         .merge(ctrl_df.alias("s"),f"t.layer = s.layer AND t.entity_name = s.entity_name"
        ).whenMatchedUpdate(set={
            "last_processed_bronze_run_id":"s.last_processed_bronze_run_id",
            "last_processed_bronze_ingestion_at":"s.last_processed_bronze_ingestion_at",
            "rows_merged":"s.rows_merged",
            "run_status":"s.run_status",
            "silver_run_id":"s.silver_run_id",
            "updated_at":"s.updated_at"
            })
         .whenNotMatchedInsertAll()
         .execute())

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at,last_run_id = get_last_processed_bronze_ingestion_at(entity_name)
    bronze_df = spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at,last_run_id
   
    return  bronze_df.filter(F.col("bronze_ingestion_at") > F.lit(last_ingested_at)), last_ingested_at, last_run_id

##Step 4 - Orders Incremental Processing

In [0]:
df_raw = spark.sql("select * from `novacart_adb`.bronze.orders_raw")
# df_raw = spark.read.table('novacart_adb.bronze.orders_raw"')
display(df_raw)

In [0]:
# Read only the bronze order rows that have not been processed yet
orders_inc, last_ingested_at,last_orders_run_id = get_incremental_bronze("novacart_adb.bronze.orders_raw","orders")
orders_inc_count = orders_inc.count()
print("Number of new orders to process :", orders_inc_count)
# Read only the bronze payment rows that have not been processed yet
if orders_inc_count > 0:
    # create a window that keep the latest order for each order_id
    order_window = Window.partitionBy("order_id").orderBy(F.col("updated_at").cast("timestamp").desc(),F.col('bronze_ingestion_at').desc())

    #start the silver order_cleaning pipeline.this block standarizes and deduplicates row order records
    orders_cleaned = (
        orders_inc
        #Standaraize order_status to uppercase so values such as shippep and SHIPPED become consistent
        .withColumn("order_status",F.upper(F.trim(F.col('order_status'))))
        .withColumn("order_status",F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        #remove formatting characters from order_amount so it can be cast to numeric type
        .withColumn("order_amount",F.regexp_replace(F.col("order_amount"),r"[$, ]",""))
        .withColumn("order_amount",F.when(F.col("order_amount").isin('N/A',"NULL","??",''),None).otherwise(F.col("order_amount")))
        .withColumn("order_amount",F.col('order_amount').cast('double'))
        .withColumn("created_at",F.to_timestamp(F.col("created_at")))
        .withColumn("updated_at",F.to_timestamp(F.col("updated_at")))
        #Assign a row number inside each business key so we ccan keep only the latest version of that record
        .withColumn("row_number",F.row_number().over(order_window))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .dropDuplicates()
        .withColumn('silver_run_id',F.lit(silver_run_id))
    )
    # Merge the cleaned orders into the silver dataset into its Delta target table
    upsert_to_silver(orders_cleaned,"novacart_adb.silver.orders_cleaned","order_id")

    # apply Silver data_quality rules to the cleaned order records
    orders_validated = (
        orders_cleaned
        .withColumn(
            "verified_by_orders_team",
            F.when(F.col("customer_id").isNull(),"verfiy customer_id")
            .when(F.col("order_amount").isNull()| (F.col("order_amount") <= 0),"verify order_amount")
            .when(F.col("order_status").isNull()| (F.trim(F.col("order_status")) == ""),"verify order_status")
            .when(F.col("product_id").isNull(),"verify product_id")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0),F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date",F.to_date("created_at"))
        .withColumn("order_month",F.month("created_at"))
        .withColumn("order_year",F.year("created_at"))
        .withColumn("order_day",F.dayofmonth("created_at"))
        .withColumn("order_dow",F.date_format("created_at",'E'))
    )

    # keep only valid order rows for the transformed silver table
    orders_good = orders_validated.filter(F.col("verified_by_orders_team") == "No Issues")
    # orders_good_count = orders_good.count()
    # print("Number of valid orders :", orders_good_count)

    orders_bad = (
        orders_validated
        .filter(F.col("verified_by_orders_team") != "No Issues")
        .withColumn("quartine_ts",F.current_timestamp())
    )
    # orders_bad_count = orders_bad.count()
    # print("Number of invalid orders :", orders_bad_count)

    # Merge the validated orders into the silver dataset into its Delta target table
    upsert_to_silver(orders_good,"novacart_adb.silver.orders_transformed","order_id")

    # Append bad rows to the quarntine table instead of losinn them
    orders_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver.orders_quarntine")

    mx_ingested = orders_inc.agg(F.max("bronze_ingestion_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingestion_at") == mx_ingested)
        .select(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    upsert_silver_control("orders",mx_run,mx_ingested,orders_good.count())

else:
    print("No new orders bronze rows for silver")
    upsert_silver_control("orders",last_orders_run_id,last_ingested_at,orders_inc_count)

In [0]:
%sql
select * from novacart_adb.silver.orders_cleaned

In [0]:
%sql
select * from novacart_adb.silver.orders_transformed

## Step 5 - Products Incremental Processing

In [0]:
%sql
select * from novacart_adb.bronze.products_raw

In [0]:
# Read only the bronze products rows that have not been processed yet
products_inc, last_product_ingested_at,last_products_run_id = get_incremental_bronze("novacart_adb.bronze.products_raw","products")
products_inc_count = products_inc.count()
print("Number of new orders to process :", products_inc_count)
# Read only the bronze product rows that have not been processed yet
if products_inc_count > 0:
    # create a window that keep the latest product for each product_id
    product_window = Window.partitionBy("product_id").orderBy(F.col("updated_at").cast("timestamp").desc(),F.col('bronze_ingestion_at').desc())

    #start the silver order_cleaning pipeline.this block standarizes and deduplicates row product records
    products_cleaned = (
        products_inc
        #Standaraize product_name to uppercase by trimming spaces and convert text into uppercase
        .withColumn("product_name",F.upper(F.trim(F.col('product_name'))))
        .withColumn("product_name",F.regexp_replace(F.col("product_name"),r"[-_]"," "))
        .withColumn("product_name",F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
        .withColumn(
            "category",
            F.when(F.upper(F.trim(F.col('category'))).contains("ELECTRNICS"),'ELECTRONICS') 
            .otherwise(F.upper(F.trim(F.col('category'))))
        )
        #start cleaning product_price field before converting it to numeric type
        .withColumn("price",F.trim(F.col('price')))
        .withColumn("price",F.regexp_replace(F.col("price"),r"\$",""))
        .withColumn("price",F.regexp_replace(F.col("price"),",","."))
        .withColumn("price",F.regexp_replace(F.col("price"),r'\s+',''))
        # .withColumn("price",F.when(F.col("price").isin('N/A','NULL','??',''),None).otherwise(F.col("price")))
        .withColumn("price",F.expr(' try_cast(price as double)'))
        .withColumn("updated_at",F.to_timestamp(F.col("updated_at")))
        #Assign a row number inside each business key so we ccan keep only the latest version of that record
        .withColumn("row_number",F.row_number().over(product_window))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .dropDuplicates()
        .withColumn('silver_run_id',F.lit(silver_run_id))
    )
    # Merge the cleaned orders into the silver dataset into its Delta target table
    upsert_to_silver(products_cleaned,"novacart_adb.silver.products_cleaned","product_id")

    # apply Silver data_quality rules to the cleaned products records
    products_validated = (
        products_cleaned
        .withColumn(
            "verified_by_products_team",
            F.when(F.col("product_name").isNull(),"verfiy product_name")
            .when(F.col("category").isNull(),"verify category")
            .when(F.col("price").isNull()| (F.col("price") <= 0),"verify price")
            .otherwise("No Issues")
        )
        .withColumn("check_product_price",
            F.when(F.col("price").isNull()| (F.col("price") <= 0),"invalid price").otherwise('valid price')
        )
    )
    # Merge the cleaned products into the silver dataset into its Delta target table

    # keep only valid product rows for the transformed silver table
    products_good = products_validated.filter(
        (F.col("verified_by_products_team") == "No Issues") & (F.col("check_product_price") == "valid price")
        )
    # products_good_count = products_good.count()
    # print("Number of valid products :", products_good_count)

    # keep valid product rows form transform silver table
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    # send invalid product rows to the quarntine dataset for manual reviews    
    products_bad = (
        products_validated
        .filter((F.col("verified_by_products_team") != "No Issues") | (F.col("check_product_price") != "valid price"))
        .withColumn("quartine_ts",F.current_timestamp())
    )
    # products_bad_count = products_bad.count()
    # print("Number of invalid products :", products_bad_count)

    # Merge the validated products into the silver dataset into its Delta target table
    upsert_to_silver(products_good,"novacart_adb.silver.products_transformed","product_id")

    # Append bad rows to the quarntine table instead of losinn them
    products_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver.products_quarntine")

    mx_ingested = products_inc.agg(F.max("bronze_ingestion_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        products_inc.filter(F.col("bronze_ingestion_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    )
    upsert_silver_control("products",mx_run,mx_ingested,products_good.count())

else:
    print("No new products bronze rows for silver")
    upsert_silver_control("products",last_products_run_id,last_product_ingested_at,products_inc_count)

In [0]:
%sql
select * from novacart_adb.silver.products_cleaned

## Step 6 : Payments Incremental Processing

In [0]:
# Read only the bronze payments rows that have not been processed yet
payments_inc, last_payments_ingested_at,last_payments_run_id  = get_incremental_bronze("novacart_adb.bronze.payments_raw","payments")
payments_inc_count = payments_inc.count()
print("Number of new payments to process :", payments_inc_count)
# display(payments_inc)
# Read only the bronze payment rows that have not been processed yet
if payments_inc_count > 0:
    # create a window that keep the latest payments for each product_id
    payments_window = Window.partitionBy("payment_id").orderBy(F.col("processed_at").cast("timestamp").desc(),F.col('bronze_ingestion_at').desc())
    
    payments_cleaned = (
        payments_inc
        .withColumn("payment_status",F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status",F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status")))
        #start cleaning paid_amount field before converting it to numeric type
        .withColumn("paid_amount",F.trim(F.col('paid_amount')))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r"\$",""))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),",","."))
        .withColumn("paid_amount",F.regexp_replace(F.col("paid_amount"),r'\s+',''))
        .withColumn("paid_amount",F.expr('try_cast(paid_amount as double)'))
        .withColumn("processed_at",F.to_timestamp(F.col("processed_at")))
        .withColumn("row_rank",F.row_number().over(payments_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .dropDuplicates()
        .withColumn("silver_run_id",F.lit(silver_run_id))
        )
     # Merge the cleaned payments into the silver dataset into its Delta target table
    upsert_to_silver(payments_cleaned,"novacart_adb.silver.payments_cleaned","payment_id")
    
    # apply Silver data_quality rules to the cleaned products records
    payments_validated = (
        payments_cleaned
        .withColumn(
            "verified_by_payments_team",
            F.when(F.col("order_id").isNull(),"verfiy order_id")
            .when(F.col("payment_status").isNull(),"verify payment_status")
            .when(F.col("paid_amount").isNull()| (F.col("paid_amount") <= 0),"verify paid_amount")
            .otherwise("No Issues")
        )
        .withColumn("check_paid_amount",
            F.when(F.col("paid_amount").isNull()| (F.col("paid_amount") <= 0),F.lit(True))
            .otherwise(F.lit(False))
        )
    )
    # keep only valid payments rows for the transformed silver table
    payments_good = payments_validated.filter((F.col("verified_by_payments_team") == "No Issues"))

    # send invalid payments rows to the quarntine dataset for manual reviews    
    payments_bad = (
        payments_validated
        .filter((F.col("verified_by_payments_team") != "No Issues"))
        .withColumn("quartine_ts",F.current_timestamp())
    )

    # Merge the validated payments into the silver dataset into its Delta target table
    upsert_to_silver(payments_good,"novacart_adb.silver.payments_transformed","payment_id")

    # Append bad rows to the quarntine table instead of losing them
    payments_bad.write.format('delta').mode("append").saveAsTable("novacart_adb.silver.payments_quarntine")

    mx_ingested = payments_inc.agg(F.max("bronze_ingestion_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        payments_inc.filter(F.col("bronze_ingestion_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    )
    upsert_silver_control("payments",mx_run,mx_ingested,payments_good.count())
else:
    print("No new payments bronze rows for silver")
    upsert_silver_control("payments",last_payments_run_id,last_payments_ingested_at,payments_inc_count)
        
    
    

# step 7 : Quick Validation

In [0]:

print("products_transformed",spark.sql("select count(*) from novacart_adb.silver.products_transformed").collect()[0][0])
print("payments_transfomred",spark.sql("select count(*) from novacart_adb.silver.payments_transformed").collect()[0][0])
print("orders_transformed",spark.sql("select count(*) from novacart_adb.silver.orders_transformed").collect()[0][0])

display(spark.table("novacart_adb.silver.processing_control").orderBy("entity_name"))